# WarehouseRobotEnv – Sprawozdanie

## Autorzy: Konrad Małek, Bartłomiej Mazgaj

## Opis problemu i celu projektu

Projekt realizuje **własne środowisko Gymnasium** o nazwie `WarehouseRobotEnv`, w którym agent steruje robotem poruszającym się po magazynie 2D.  
Robot zbiera paczki rozmieszczone na mapie i dostarcza je do strefy bazy, unikając przy tym prostokątnych przeszkód (regałów) i ścian.

### Kluczowe właściwości środowiska

| Właściwość | Wartość |
|---|---|
| Przestrzeń akcji | ciągła `Box([-1, 1]², float32)` – przyspieszenie (ax, ay) |
| Przestrzeń obserwacji | ciągła `Box([-1, 1]^17, float32)` – 17 cech |
| Mapa | kwadrat 10×10 jednostek, 2 prostokątne przeszkody |
| Liczba paczek | 3 (do zebrania i dostarczenia do bazy) |
| Koniec epizodu | sukces (wszystkie paczki) lub limit 500 kroków |

Do rozwiązania problemu użyto algorytmu **PPO** z biblioteki `stable-baselines3`.

## 1. Środowisko – przestrzeń akcji, obserwacji i przykładowy krok

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from env.warehouse_robot_env import WarehouseRobotEnv

env = WarehouseRobotEnv()
print('action_space  :', env.action_space)
print('observation_space:', env.observation_space)

obs, info = env.reset(seed=42)
print('\nPrzykładowa obserwacja po reset():')
labels = [
    'x', 'y', 'vx', 'vy', 'carrying',
    'dx_target', 'dy_target', 'dist_target',
    *[f'sensor_{i}' for i in range(8)],
    'n_pkgs_left'
]
for i, (label, val) in enumerate(zip(labels, obs)):
    print(f'  [{i:2d}] {label:15s} = {val:.4f}')

### Opis składowych obserwacji

| Indeks | Składowa | Normalizacja |
|---|---|---|
| 0 | x – pozycja X robota | `/ map_size * 2 − 1` → `[−1, 1]` |
| 1 | y – pozycja Y robota | `/ map_size * 2 − 1` → `[−1, 1]` |
| 2 | vx – prędkość X | `/ v_max` → `[−1, 1]` |
| 3 | vy – prędkość Y | `/ v_max` → `[−1, 1]` |
| 4 | carrying – flaga niesienia paczki | 0.0 lub 1.0 |
| 5 | dx_target – wektor do celu X | `/ map_size` |
| 6 | dy_target – wektor do celu Y | `/ map_size` |
| 7 | dist_target – odległość do celu | `/ (map_size · √2)` |
| 8–15 | 8 czujników odległości (co 45°) | `/ sensor_range` → `[0, 1]` |
| 16 | n_packages_left – pozostałe paczki | `/ n_packages_initial` |

## 2. Walidacja środowiska (check_env + losowe epizody)

In [ ]:
from gymnasium.utils.env_checker import check_env

check_env(WarehouseRobotEnv(), warn=True)
print('check_env: OK')

In [ ]:
N_RANDOM = 5
rng = np.random.default_rng(0)

for ep in range(N_RANDOM):
    obs, _ = env.reset(seed=int(rng.integers(1000)))
    total_r, steps, collisions = 0.0, 0, 0
    done = False
    while not done:
        obs, r, terminated, truncated, info = env.step(env.action_space.sample())
        total_r += r
        steps += 1
        collisions += int(info['collision'])
        done = terminated or truncated
    print(f'Ep {ep+1}: steps={steps:4d}  reward={total_r:8.1f}  '
          f'delivered={info["n_deliveries_total"]}  collisions={collisions}')

## 3. Funkcja nagrody

Funkcja nagrody składa się z kilku składników, które wspólnie promują skuteczne i strategiczne zachowanie:

| Składnik | Wartość | Cel |
|---|---|---|
| Kara za krok | −0.1 | zachęca do szybkiego działania |
| Kara za kolizję | −5.0 | unikanie przeszkód i ścian |
| Podniesienie paczki | +20.0 | nagroda za zbliżenie i zwolnienie przy paczce |
| Dostarczenie do bazy | +100.0 | główny cel |
| Ukończenie wszystkich | +200.0 | bonus za pełne zadanie |
| Reward shaping | β·(d_prev − d_curr) | gęsty sygnał postępu w kierunku celu |

**Aktualny cel** zmienia się dynamicznie:
- gdy `carrying=False` → najbliższa niezebrana paczka,
- gdy `carrying=True` → baza.

Podniesienie/dostarczenie wymaga jednoczesnego **zbliżenia** (w promieniu `r_pickup`/`r_delivery`) **i zwolnienia** poniżej `v_pickup_max=0.5` – wymusza strategiczne spowolnienie przy celu.

## 4. Algorytm PPO – uzasadnienie wyboru

Do rozwiązania problemu wybrano **PPO (Proximal Policy Optimization)**:

- Naturalnie obsługuje **ciągłą przestrzeń akcji** (rozkład normalny na wyjściu aktora).
- Stabilny trening dzięki ograniczeniu kroków polityki (clipping) – nie wymaga precyzyjnego doboru hiperparametrów.
- Efektywnie korzysta z **równoległych środowisk** (VecEnv), co przyspiesza zbieranie doświadczeń.
- Dostępny gotową implementacją w `stable-baselines3`.

Użyte hiperparametry (sekcja 17.7 specyfikacji):

| Hiperparametr | Wartość |
|---|---|
| total_timesteps | 2 500 000 |
| n_envs | 4 |
| learning_rate | 3×10⁻⁴ |
| n_steps | 2048 |
| batch_size | 64 |
| gamma | 0.99 |
| gae_lambda | 0.95 |

## 5. Ewaluacja wytrenowanego modelu

In [ ]:
from stable_baselines3 import PPO

MODEL_PATH = '../models/random_v2/best_model.zip'
model = PPO.load(MODEL_PATH)
print(f'Model wczytany z {MODEL_PATH}')

In [ ]:
N_EVAL = 20
eval_env = WarehouseRobotEnv()

rewards, lengths, deliveries, collisions_ep = [], [], [], []

for ep in range(N_EVAL):
    obs, _ = eval_env.reset(seed=ep)
    total_r, steps = 0.0, 0
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, r, terminated, truncated, info = eval_env.step(action)
        total_r += r
        steps += 1
        done = terminated or truncated
    rewards.append(total_r)
    lengths.append(steps)
    deliveries.append(info['n_deliveries_total'])
    collisions_ep.append(info['n_collisions_total'])

print(f'Wyniki ewaluacji na {N_EVAL} epizodach (deterministyczna polityka):')
print(f'  Średni reward:        {np.mean(rewards):8.1f} ± {np.std(rewards):.1f}')
print(f'  Średnia długość:      {np.mean(lengths):8.1f} kroków')
print(f'  Śr. dostarczone paczki: {np.mean(deliveries):.2f} / 3')
print(f'  Śr. kolizje/epizod:   {np.mean(collisions_ep):.1f}')
print(f'  Sukces (3/3 paczek):  {sum(d == 3 for d in deliveries)}/{N_EVAL} epizodów')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

axes[0].bar(range(N_EVAL), rewards, color='steelblue')
axes[0].axhline(np.mean(rewards), color='red', linestyle='--', label=f'Śr. {np.mean(rewards):.0f}')
axes[0].set_title('Reward per episode')
axes[0].set_xlabel('Epizod'); axes[0].set_ylabel('Reward'); axes[0].legend()

axes[1].bar(range(N_EVAL), deliveries, color='green')
axes[1].axhline(np.mean(deliveries), color='red', linestyle='--', label=f'Śr. {np.mean(deliveries):.2f}')
axes[1].set_title('Dostarczone paczki / epizod')
axes[1].set_xlabel('Epizod'); axes[1].set_ylabel('Paczki'); axes[1].set_ylim(0, 3.5); axes[1].legend()

axes[2].bar(range(N_EVAL), collisions_ep, color='salmon')
axes[2].axhline(np.mean(collisions_ep), color='red', linestyle='--', label=f'Śr. {np.mean(collisions_ep):.1f}')
axes[2].set_title('Kolizje / epizod')
axes[2].set_xlabel('Epizod'); axes[2].set_ylabel('Kolizje'); axes[2].legend()

plt.tight_layout()
plt.show()

## 6. Krzywe uczenia

Krzywa uczenia pozwala ocenić, czy agent rzeczywiście poprawiał swoją politykę w trakcie treningu, a nie tylko chwilowo uzyskiwał lepsze wyniki. W analizowanym przebiegu widać wzrost średniej nagrody epizodu (`ep_rew_mean`), a następnie jej stabilizację, co sugeruje stopniowe wykształcenie skutecznej strategii działania.

Interpretacja wykresu jest następująca:
- na początku treningu reward jest niższy, ponieważ agent eksploruje środowisko i częściej wykonuje nieefektywne ruchy,
- wraz z kolejnymi aktualizacjami PPO średnia nagroda rośnie, co oznacza lepsze omijanie przeszkód, skuteczniejsze podnoszenie paczek i częstsze dostawy,
- końcowe wypłaszczenie krzywej oznacza, że polityka zbliżyła się do stabilnego zachowania i dalsze zyski z treningu były już mniejsze.

W praktyce taka krzywa jest dobrym sygnałem, że agent nie działa losowo, tylko uczy się powtarzalnej procedury: najpierw dojazdu do paczki, potem kontrolowanego spowolnienia przy podniesieniu, a na końcu dostarczenia jej do strefy docelowej.

In [ ]:
import glob
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

LOG_DIR = '../logs/random_v2'
event_files = glob.glob(os.path.join(LOG_DIR, '**', 'events.out.tfevents.*'), recursive=True)

if not event_files:
    print('Brak plików TensorBoard – uruchom trening przed generowaniem wykresów.')
else:
    # Wybierz plik z katalogu eval (tam jest ep_rew_mean)
    # Preferuj katalog ppo_warehouse_*/ (głowny trening)
    train_files = [f for f in event_files if 'eval' not in f]
    eval_files  = [f for f in event_files if 'eval' in f]

    def load_scalar(files, tag):
        for f in sorted(files):
            ea = EventAccumulator(f)
            ea.Reload()
            if tag in ea.Tags().get('scalars', []):
                events = ea.Scalars(tag)
                steps = [e.step for e in events]
                vals  = [e.value for e in events]
                return steps, vals
        return None, None

    plots = []

    # Krzywa treningowa pokazuje, czy agent systematycznie poprawiał politykę.
    steps, vals = load_scalar(train_files, 'rollout/ep_rew_mean')
    if steps:
        plots.append(('Reward (trening)', 'ep_rew_mean', steps, vals, 'steelblue'))

    # Wykres ewaluacyjny dodajemy tylko wtedy, gdy rzeczywiście istnieją logi eval.
    steps, vals = load_scalar(eval_files, 'eval/mean_reward')
    if steps:
        plots.append(('Reward (ewaluacja)', 'mean_reward', steps, vals, 'green'))

    if not plots:
        print('Brak dostępnych danych scalar do narysowania krzywej uczenia.')
    else:
        fig, axes = plt.subplots(1, len(plots), figsize=(6 * len(plots), 4))
        if len(plots) == 1:
            axes = [axes]

        for ax, (title, ylabel, steps, vals, color) in zip(axes, plots):
            ax.plot(steps, vals, color=color)
            ax.set_title(title)
            ax.set_xlabel('Timestep')
            ax.set_ylabel(ylabel)
            ax.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()

        

## 7. Eksperymenty

Najważniejszy eksperyment polegał na porównaniu wytrenowanego agenta PPO z polityką losową. To porównanie jest istotne z punktu widzenia treści zadania, ponieważ pozwala sprawdzić, czy akcje agenta wynikają z wyuczonej strategii, a nie z przypadku.

Jeżeli agent działałby losowo, jego średni reward oraz liczba dostarczonych paczek byłyby zbliżone do wyników polityki losowej. Otrzymane wyniki pokazują jednak dużą różnicę jakościową: PPO regularnie realizuje cel środowiska, natomiast polityka losowa praktycznie nie doprowadza do skutecznych dostaw.

Wniosek z eksperymentu jest więc następujący: sekwencja akcji wykonywana przez agenta wskazuje na działanie według strategii. Agent najpierw zbliża się do paczki, następnie redukuje prędkość przy podniesieniu, omija przeszkody i kieruje się do strefy dostawy. Taki schemat zachowania jest powtarzalny i celowy, więc nie można go uznać za losowy.

In [ ]:
# Eksperyment A: agent wytrenowany vs losowa polityka
def run_policy(policy_fn, n_episodes=10, seed_offset=100):
    e = WarehouseRobotEnv()
    rs, ds = [], []
    for i in range(n_episodes):
        obs, _ = e.reset(seed=seed_offset + i)
        total_r, done = 0.0, False
        while not done:
            action = policy_fn(obs)
            obs, r, terminated, truncated, info = e.step(action)
            total_r += r
            done = terminated or truncated
        rs.append(total_r)
        ds.append(info['n_deliveries_total'])
    return np.mean(rs), np.mean(ds)

ppo_r, ppo_d = run_policy(lambda obs: model.predict(obs, deterministic=True)[0])
rand_r, rand_d = run_policy(lambda obs: eval_env.action_space.sample())

print('=== Agent PPO vs losowa polityka ===')
print(f'PPO:  reward={ppo_r:.1f}  dostarczone={ppo_d:.2f}/3')
print(f'Rand: reward={rand_r:.1f}  dostarczone={rand_d:.2f}/3')

In [ ]:
# Wizualizacja trajektorii agenta PPO (rgb_array)
viz_env = WarehouseRobotEnv(render_mode='rgb_array')
obs, _ = viz_env.reset(seed=7)
frames = []
done = False
while not done and len(frames) < 400:
    action, _ = model.predict(obs, deterministic=True)
    obs, _, terminated, truncated, _ = viz_env.step(action)
    done = terminated or truncated
    frame = viz_env.render()
    if frame is not None:
        frames.append(frame)
viz_env.close()

print(f'Zebranych klatek: {len(frames)}')

# Wyświetl co N-tą klatkę (siatka: kilka wierszy/kolumn)
show_every = max(1, len(frames) // 16)
selected = frames[::show_every][:16]  # pokaż do 16 klatek domyślnie
# Konfigurowalne: liczba kolumn w siatce (zmień według potrzeby)
cols = min(4, len(selected))
import math
rows = math.ceil(len(selected) / cols) if cols>0 else 1
# Każdy podwykres będzie około 4x4 cala - dostosuj mnożnik aby powiększyć
fig_w = cols * 4
fig_h = rows * 4
fig, axes = plt.subplots(rows, cols, figsize=(fig_w, fig_h))
# Ujednolić axes do płaskiej listy
if isinstance(axes, np.ndarray):
    axes = axes.flatten().tolist()
else:
    axes = [axes]
# Wypełnij siatkę obrazami; puste pola wyłącz
for i, ax in enumerate(axes):
    if i < len(selected):
        ax.imshow(selected[i])
        ax.axis('off')
    else:
        ax.axis('off')
plt.suptitle('Trajektoria agenta PPO (wybrane klatki)', y=1.02, fontsize=16)
plt.tight_layout()
plt.subplots_adjust(top=0.90)
plt.show()

## 8. Wnioski końcowe

1. **Środowisko** `WarehouseRobotEnv` spełnia wszystkie wymagania API Gymnasium (`check_env` bez błędów), używa ciągłej przestrzeni akcji i obserwacji oraz implementuje tryb graficzny (`render_mode='rgb_array'` / `'human'`).

2. **Algorytm PPO** skutecznie nauczył się strategii zbierania i dostarczania paczek. Agent unika przeszkód, planuje kolejność zbierania paczek i zwalnia przy celach – zachowanie zgodne z reward shapingiem.

3. **Reward shaping** (β·Δd) zapewnił gęsty sygnał uczenia, który przyspiesza zbieżność w porównaniu do samych rzadkich nagród za podniesienie/dostarczenie.

4. **Warunek prędkości** przy podnoszeniu/dostarczaniu (`v < v_pickup_max`) wymusił na agencie nieintuicyjne zachowanie – celowe spowalnianie przy celach – co jest widoczne w trajektoriach.

5. Porównanie z losową polityką potwierdza, że agent działa **zgodnie ze strategią**, a nie losowo. Sekwencja akcji jest ukierunkowana na realizację celu: agent wybiera paczki, dojeżdża do nich, odpowiednio zwalnia przy podniesieniu i finalnie transportuje je do obszaru dostawy.

## 9. Wizualizacja działania agenta

Poniższa komórka uruchamia wizualizację analogiczną do skryptu `play_agent.py`. Agent działa w trybie `human`, więc można obserwować jego zachowanie w oknie `pygame` i sprawdzić, czy realizuje strategię w sposób powtarzalny.

In [ ]:
import os
import sys
import time
import pygame
from stable_baselines3 import PPO

ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from env.warehouse_robot_env import WarehouseRobotEnv

MODEL_PATH = os.path.join(ROOT, 'models', 'random_v2', 'best_model.zip')
N_EPISODES = 5
RANDOM_LAYOUT = True

env = WarehouseRobotEnv(render_mode='human', random_layout=RANDOM_LAYOUT)
model = PPO.load(MODEL_PATH, env=env)

for ep in range(N_EPISODES):
    obs, info = env.reset()
    env.render()  # inicjalizacja pygame przed pierwszym event.get()
    total_reward = 0.0
    step = 0
    done = False

    while not done:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                env.close()
                raise SystemExit('Zamknięto okno wizualizacji.')

        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step += 1
        done = terminated or truncated
        env.render()

    print(
        f'Episode {ep + 1}: steps={step}, reward={total_reward:.1f}, '
        f"delivered={info['n_deliveries_total']}, "
        f"collisions={info['n_collisions_total']}"
    )
    time.sleep(1.0)

env.close()